# 01 - Exploratory Data Analysis

This notebook produces the EDA story for the CS-483 Urban Crime Intelligence project.

We answer, in order:

1. How large and how balanced is the cleaned dataset?
2. What are the dominant crime types and locations?
3. When do crimes happen (hour, weekday, month, year)?
4. Where do crimes concentrate geographically?
5. Do numeric features correlate with the Arrest target?

All figures are produced from `data/processed/chicago_cleaned.csv`, which is
created by running `python -m src.main` (or just the `clean` step).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

df = pd.read_csv("../data/processed/chicago_cleaned.csv")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "->", df["Date"].max())
print("Columns:", list(df.columns))
df.head()

## 1. Arrest class balance

The Arrest column is the supervised target. If it is heavily imbalanced
we need to compensate during modeling (class weights, SMOTE, or both).

In [ ]:
arrest_counts = df["Arrest"].value_counts()
arrest_rate = df["Arrest"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=["No Arrest (0)", "Arrest (1)"], y=arrest_counts.values, ax=ax)
for i, v in enumerate(arrest_counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title(f"Arrest class distribution (arrest rate = {arrest_rate:.2%})")
ax.set_ylabel("Records")
plt.tight_layout()
plt.show()

print(f"Arrest rate: {arrest_rate:.4f}")
print(f"Imbalance ratio (No Arrest : Arrest): {arrest_counts[0] / arrest_counts[1]:.2f}:1")

## 2. Top crime types and locations

These two distributions decide which categorical encodings will dominate
the Random Forest feature importance ranking later.

In [ ]:
top_types = df["Primary Type"].value_counts().head(10)
top_locs = df["Location Description"].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=top_types.values, y=top_types.index, ax=axes[0], orient="h")
axes[0].set_title("Top 10 Primary Crime Types")
axes[0].set_xlabel("Incidents")

sns.barplot(x=top_locs.values, y=top_locs.index, ax=axes[1], orient="h")
axes[1].set_title("Top 10 Location Descriptions")
axes[1].set_xlabel("Incidents")
plt.tight_layout()
plt.show()

print("Arrest rate by top crime type:")
print(df[df["Primary Type"].isin(top_types.index)].groupby("Primary Type")["Arrest"].mean().sort_values(ascending=False))

## 3. Temporal patterns

Hour-of-day, weekday, month, and year. These drive the `Hour`, `DayOfWeek`,
`IsWeekend`, `IsNight`, `Month`, `Season`, and `Year` engineered features.

In [ ]:
df["Hour"] = df["Date"].dt.hour
df["DayOfWeek"] = df["Date"].dt.day_name()
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

hourly = df["Hour"].value_counts().sort_index()
axes[0, 0].bar(hourly.index, hourly.values, color="#1f77b4")
axes[0, 0].set_title("Incidents by hour of day")
axes[0, 0].set_xlabel("Hour")
axes[0, 0].set_ylabel("Incidents")

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = df["DayOfWeek"].value_counts().reindex(weekday_order)
axes[0, 1].bar(weekday.index, weekday.values, color="#ff7f0e")
axes[0, 1].set_title("Incidents by day of week")
axes[0, 1].tick_params(axis="x", rotation=30)

monthly = df["Month"].value_counts().sort_index()
axes[1, 0].bar(monthly.index, monthly.values, color="#2ca02c")
axes[1, 0].set_title("Incidents by month")
axes[1, 0].set_xlabel("Month")

yearly = df["Year"].value_counts().sort_index()
axes[1, 1].plot(yearly.index, yearly.values, marker="o", color="#d62728")
axes[1, 1].set_title("Incidents per year")
axes[1, 1].set_xlabel("Year")

plt.tight_layout()
plt.show()

print("Arrest rate by hour (top 5 highest):")
print(df.groupby("Hour")["Arrest"].mean().sort_values(ascending=False).head(5))

## 4. Geographic spread

A random 10k-point scatter is enough to visually confirm that all records
fall inside the Chicago bounding box applied during cleaning.

In [ ]:
geo_sample = df.sample(n=min(10_000, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(
    geo_sample["Longitude"],
    geo_sample["Latitude"],
    s=2,
    alpha=0.3,
    c=geo_sample["Arrest"].map({0: "#1f77b4", 1: "#d62728"}),
)
ax.set_title("Chicago incident scatter (red = arrest)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

## 5. Correlation with the Arrest target

Numeric-only correlation heatmap. We expect low linear correlations here,
which motivates using non-linear models (Random Forest, SVM-RBF) rather
than relying on Logistic Regression alone.

In [ ]:
num_cols = ["Hour", "Month", "Year", "District", "Community Area", "Latitude", "Longitude", "Arrest"]
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Numeric feature correlation")
plt.tight_layout()
plt.show()

## EDA Takeaways

- **Severe class imbalance.** Arrest rate is well below 50%, so models must use
  `class_weight="balanced"` or SMOTE or both to avoid trivial majority-class predictions.
- **Crime type and location carry strong signal.** The top 10 primary types and
  location descriptions concentrate most of the data; the arrest rate swings
  substantially across them, so the encoded categorical features will likely
  dominate Random Forest feature importance.
- **Clear temporal structure.** Incidents cluster in late-night hours, on weekends,
  and in warmer months. The `IsNight`, `IsWeekend`, and `Season` engineered
  features exist to expose this to downstream models.
- **Weak linear correlations with Arrest.** The Pearson correlation heatmap
  shows no single numeric predictor strongly aligned with the target; this is
  why non-linear models (Random Forest, SVM-RBF) are expected to outperform
  plain Logistic Regression.